# 新融合：SAMMLV 首轮筛选
沿用已可运行的 ME-TST+ Colab 环境。按顺序运行下面三个 cell；第一个上传本次 ZIP 包。
此 notebook 不安装环境，不训练模型，不生成官方响应。

In [ ]:
from google.colab import drive, files
from pathlib import Path
import io, zipfile, subprocess, sys, os
from datetime import datetime, timezone

drive.mount('/content/drive')
uploaded = files.upload()  # 上传 metst_agreement_bonus.zip
name = 'metst_agreement_bonus.zip'
assert name in uploaded, '请选择 metst_agreement_bonus.zip'
task_dir = Path('/content/glsd_agreement_bonus_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%f'))
task_dir.mkdir(exist_ok=False)
expected = {'run_agreement_bonus.py', 'run_generalized_mean_screening.py',
            'official_response_component_ablation.py', 'colab_entry.py',
            'test_agreement_bonus.py', 'README.md'}
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
    assert set(z.namelist()) == expected, '上传包内容不匹配'
    z.extractall(task_dir)
run_env = dict(os.environ)
run_env['PYTHONPATH'] = os.pathsep.join(dict.fromkeys(
    [str(task_dir), '/content/ME-TST', os.getcwd()] +
    [p for p in sys.path if p] + [run_env.get('PYTHONPATH', '')]))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(task_dir),
                '-p', 'test_agreement_bonus.py', '-v'], env=run_env, check=True)
subprocess.run([sys.executable, str(task_dir / 'colab_entry.py'),
                '--setting', 'sammlv', '--check-inputs'], env=run_env, check=True)
print('CELL_1_READY = PASS')


In [ ]:
run_tag = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_base = Path('/content/drive/MyDrive/GLSD_AGREEMENT_BONUS')
output_base.mkdir(parents=True, exist_ok=True)
result_dir = output_base / ('sammlv_' + run_tag)
log_path = output_base / ('sammlv_' + run_tag + '.log')
command = [sys.executable, '-u', str(task_dir / 'colab_entry.py'),
           '--setting', 'sammlv', '--output', str(result_dir)]
print('结果目录:', result_dir)
print('日志:', log_path)
with log_path.open('x', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, env=run_env)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        raise
assert returncode == 0, f'运行失败，请提供日志：{log_path}'
assert (result_dir / 'completion.json').is_file(), '没有完成标志，请检查日志'
print('CELL_2_COMPLETE = PASS')


In [ ]:
import pandas as pd
import shutil
summary = pd.read_csv(result_dir / 'screening_summary_full.csv')
display(summary[~summary['variant'].str.startswith('fixed_')])
display(pd.read_csv(result_dir / 'metst_sammlv' / 'paired_comparisons.csv'))
archive = shutil.make_archive('/content/' + result_dir.name + '_results',
                              'zip', root_dir=str(result_dir))
files.download(archive)
